## Assignment 1: Prompting for Program Repair

In this assignment, you will design prompts to repair buggy Python functions and evaluate the quality of the bug repair using unit tests.

What you will do:
- Use this notebook as a guide to complete your assignment.
- Inside the `data` directory, you’ll find one folder per buggy Python function (`data/BUGxxx/`), each with example tests in `test_bug.py`.
- Experiment with prompting techniques using the model (Gemma 3 270M IT), and observe how variations in prompt wording affect bug repair performance.
- For each buggy Python function, record the prompts you used, then choose one final repaired code to submit.

Submission Instructions:  
Submit a ZIP file of the `results` folder, containing the two items listed below, named `SWE270P_A1.ZIP`.  
**Please make sure to strictly follow the submission format and filename, as an autograder will be used for grading. <br> An incorrect format or filename may result in a score of 0.**  
 1) `prompts.jsonl`: all the prompts you used. This notebook includes a helper function that logs them to `results/prompts.jsonl`.
 2) `BUGxxx.py` files: one final repaired code file per buggy Python function (e.g., `BUG001.py`), saved under `results/final_patches/`.  


Required File Structure:  
Your submission ZIP file **must** follow this exact file structure for compatibility with the autograder:    
```
SWE270P_A1.ZIP
└── results
    ├── prompts.jsonl
    └── final_patches
        ├── BUG000.py
        ├── ...
        └── BUG009.py
```

Helpful Guidelines:
- Use `test_bug.py` to evaluate the quality of repaired code.
- Add your own test cases to check the robustness of your repaired code.
- Final submissions will be graded using an autograder. Do not modify any function names or parameter definitions.
- Each final patch must be a complete replacement for that bug's `buggy.py`. Keep its imports and helper functions, not only the fixed function (e.g., BUG001 needs `get` and `insert_or_update`).

Recommended process:
1) Work on one bug at a time (e.g., BUG001).
2) Select a prompting technique (e.g., zero-shot, chain-of-thought, tests-aware, or self-refine).
3) Using the selected technique, draft your prompt and call the model.
4) Copy the repaired code into that bug's `buggy.py` and run its tests with `test_bug.py` (e.g., in `data/BUG001/`). <br>
    **For example**, for BUG001, run `pytest test_bug.py` in a terminal inside `data/BUG001/`, or run `!cd data/BUG001 && pytest test_bug.py` in a notebook cell.
5) Modify the prompt using different wording and structure, or try another prompting technique until the repaired code passes all tests.   
6) Log the prompting technique and the prompt used to call the model in `prompts.jsonl`.
7) Save your final repaired code for each bug (e.g., `BUG001.py`) for submission.



## How to run this notebook

1. Open it in **Google Colab** and pick a GPU runtime (Runtime → Change runtime type → **T4 GPU**).
2. Upload the course ZIP (`SWE270P_A1.zip`, the one with `data/BUG000` … `data/BUG009`) in the Colab **Files** panel. The *Locate the assignment data* cell below unzips it and moves into that folder.
3. Request access on the [Gemma 3 270M IT model page](https://huggingface.co/google/gemma-3-270m-it) and have a Hugging Face **Read** token ready for `login()`.
4. **Runtime → Run all.** The notebook runs the four technique demos, then the automatic repair loop over all ten bugs. Every prompt it sends is logged to `results/prompts.jsonl`, and the best tested patch for each bug is saved to `results/final_patches/BUGxxx.py`.
5. The last cell checks the folder structure, builds **`SWE270P_A1.ZIP`** and downloads it.

The original files in `data/` are never modified. Each candidate is tested in a scratch copy under `work/`.


## 0. Setup (Required: Gemma 3 270M IT)

This assignment **must use** `google/gemma-3-270m-it` via Hugging Face Transformers (local CPU/GPU). No external APIs are needed.

We recommend using Google Colab. If you're using Colab, make sure you have a GPU runtime (Runtime → Change runtime type → select T4 GPU). If you run locally, use Python 3.10 or newer.

In [ ]:
# Run this only if you are running locally
%pip install transformers ipywidgets torch accelerate

In [ ]:
%pip install pytest huggingface_hub

Note:

1. To use Hugging Face models, you need a Hugging Face access token (see [User access tokens](https://huggingface.co/docs/hub/en/security-tokens)).  
A **Read** token works. If you create a fine-grained token instead, check the box for "Read access to contents of all public gated repos you can access". Paste the token into the code below when prompted.  

2. Gemma 3 is a gated model, so you need to request access on the [model page](https://huggingface.co/google/gemma-3-270m-it) before downloading it.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Load model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-270m-it",
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

def generate_text(prompt: str, max_new_tokens: int = 200, do_sample: bool = False,
                  temperature: float = 0.7, top_p: float = 0.95) -> str:
    """Send `prompt` as one user turn to gemma-3-270m-it and return only the newly generated text.

    Decoding is greedy by default so a prompt always gives the same answer; pass do_sample=True
    to draw a different candidate when retrying the same prompt.
    """
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    gen_kwargs = {"max_new_tokens": max_new_tokens, "do_sample": do_sample}
    if do_sample:
        gen_kwargs.update(temperature=temperature, top_p=top_p, top_k=64)

    with torch.inference_mode():
        output_ids = model.generate(**inputs, **gen_kwargs)

    # Decode only the tokens after the prompt; skip_special_tokens drops the trailing <end_of_turn>.
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Check if the model is running:
print(generate_text("Hello from Gemma-3-it!"))


### Locate the assignment data
Finds the folder that contains `data/BUG000`, unzipping the uploaded course ZIP first if needed, and makes it the working directory. `results/` is then created next to `data/`.

In [ ]:
import os, glob, zipfile
from pathlib import Path

# Colab's Files panel can upload to /content (the default) or, if you were browsing "..", to /.
SEARCH_DIRS = [Path("."), Path("/content"), Path("/")]

def _find_assignment_root():
    for base in SEARCH_DIRS:
        for cand in (base, base / "SWE270P_A1"):
            if (cand / "data" / "BUG000").is_dir():
                return cand.resolve()
    # Not unzipped yet: find the uploaded course ZIP and extract it (into /content on Colab).
    for base in SEARCH_DIRS:
        for z in sorted(base.glob("*.zip")):
            with zipfile.ZipFile(z) as zf:
                hits = [n for n in zf.namelist()
                        if n.endswith("data/BUG000/buggy.py") and "__MACOSX" not in n]
                if hits:
                    target = Path("/content") if Path("/content").is_dir() else z.parent
                    zf.extractall(target)
                    print(f"Extracted {z} into {target}")
                    return (target / hits[0][:-len("data/BUG000/buggy.py")]).resolve()
    raise FileNotFoundError(
        "Could not find data/BUG000. Upload the course ZIP (with data/BUG000 ... data/BUG009) "
        "in the Colab Files panel, then re-run this cell."
    )

os.chdir(_find_assignment_root())
print("Working directory:", os.getcwd())
print("Bugs found:", sorted(p.name for p in Path("data").glob("BUG*")))


## Prompting Techniques: Zero‑Shot, One‑Shot, Chain‑of‑Thought, and Few‑Shot  
**Goal:** Practice different prompting strategies and compare their effects on bug repair correctness.

After generating the fix for a problem, you can paste it into the corresponding `buggy.py` and run `pytest` for that folder to see if it passes.

In [ ]:
from pathlib import Path
import json, re, ast, textwrap, datetime as dt
from typing import Optional

"""
Helper function to save the prompts.jsonl file.
Helper functions to extract the repaired code from the LLM response and save the complete file.
"""

PROMPTS_PATH = Path("results/prompts.jsonl")
PATCH_DIR = Path("results/final_patches")                        # Use this folder to save the repaired Python code
PROMPTS_PATH.parent.mkdir(parents=True, exist_ok=True)
PATCH_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("data")
MODEL_ID = "google/gemma-3-270m-it"

# The function each bug's test_bug.py imports, i.e. the one the model has to repair.
TARGET_FUNCTION = {
    "BUG000": "has_close_elements",
    "BUG001": "shortest_path_length",
    "BUG002": "shortest_paths",
    "BUG003": "minimum_spanning_tree",
    "BUG004": "shortest_path_lengths",
    "BUG005": "breadth_first_search",
    "BUG006": "depth_first_search",
    "BUG007": "reverse_linked_list",
    "BUG008": "detect_cycle",
    "BUG009": "longestValidParentheses",
}

def log_prompt(technique: str, prompt: str,
               completion: Optional[str] = None,
               bug_file: Optional[str] = None,
               model: Optional[str] = None,
               ):
    rec = {
        "timestamp": dt.datetime.now().isoformat(timespec="seconds"),
        "technique": technique,
        "prompt": prompt,
    }
    if completion is not None: rec["completion"] = completion
    if bug_file is not None:   rec["bug_file"] = bug_file
    if model is not None:      rec["model"] = model
    with PROMPTS_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


# A fenced block; an unterminated one (generation hit max_new_tokens) runs to the end of the text.
_CODE_BLOCK = re.compile(r"```[ \t]*(?:python|py|Python)?[ \t]*\n(.*?)(?:```|\Z)", re.DOTALL)

def extract_code(model_output: str, func_name: Optional[str] = None) -> Optional[str]:
    """Pull the Python code out of a model response.

    Prefers the last fenced block that defines `func_name` (in chain-of-thought answers the
    final block is the fix), then the longest fenced block, then everything from the first `def`.
    """
    blocks = [textwrap.dedent(b).strip() for b in _CODE_BLOCK.findall(model_output)]
    blocks = [b for b in blocks if b]
    if func_name:
        defining = [b for b in blocks if re.search(rf"^\s*def\s+{func_name}\s*\(", b, re.M)]
        if defining:
            return defining[-1]
    if blocks:
        return max(blocks, key=len)
    m = re.search(r"^def\s+\w+\s*\(", model_output, re.M)
    return model_output[m.start():].strip() if m else None


def _parse_lenient(code: str):
    """ast.parse, dropping trailing lines (e.g. prose after an unfenced function) until it parses."""
    for _ in range(30):
        try:
            return code, ast.parse(code)
        except SyntaxError as e:
            if not e.lineno or e.lineno <= 1:
                return None
            code = "\n".join(code.splitlines()[:e.lineno - 1])
    return None


def _function_node(tree, name):
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name == name:
            return node
    return None


def merge_into_original(original_source: str, model_code: str, func_name: str) -> Optional[str]:
    """Replace only `func_name` in the original buggy.py with the model's version.

    Imports, helper functions (e.g. BUG001's get / insert_or_update) and the module docstring are
    kept from the original, so the result is a complete drop-in replacement for buggy.py.
    Returns None if the model's code has no usable function or changes the parameters.
    """
    parsed = _parse_lenient(model_code)
    if parsed is None:
        return None
    code, tree = parsed
    new_fn = _function_node(tree, func_name)
    if new_fn is None:
        funcs = [n for n in tree.body if isinstance(n, ast.FunctionDef)]
        if len(funcs) != 1:
            return None
        new_fn = funcs[0]  # the model renamed the function; the name is restored below

    old_fn = _function_node(ast.parse(original_source), func_name)
    if [a.arg for a in new_fn.args.args] != [a.arg for a in old_fn.args.args]:
        return None  # the autograder calls the original signature

    new_lines = code.splitlines()[new_fn.lineno - 1:new_fn.end_lineno]
    new_lines[0] = re.sub(r"def\s+\w+", f"def {func_name}", new_lines[0], count=1)
    orig_lines = original_source.splitlines()
    merged = "\n".join(orig_lines[:old_fn.lineno - 1] + new_lines + orig_lines[old_fn.end_lineno:]) + "\n"
    try:
        ast.parse(merged)
    except SyntaxError:
        return None
    return merged


def save_final_patch(bug_file: str, model_output: str) -> str:
    """bug_file is a file name like 'BUG001.py'. Saves to results/final_patches/BUG001.py

    Save the complete file (imports and helper functions included), not only the fixed function.
    """
    bug_id = Path(bug_file).stem
    func_name = TARGET_FUNCTION.get(bug_id)
    code = extract_code(model_output, func_name)
    if code is None:
        raise ValueError(f"No Python code found in the model output for {bug_file}")

    original_path = DATA_DIR / bug_id / "buggy.py"
    if func_name and original_path.exists():
        merged = merge_into_original(original_path.read_text(encoding="utf-8"), code, func_name)
        if merged is None:
            raise ValueError(f"The model output for {bug_file} has no usable `{func_name}` function")
        code = merged

    out_path = PATCH_DIR / bug_file
    out_path.write_text(code if code.endswith("\n") else code + "\n", encoding="utf-8")
    return str(out_path)


### Repair toolkit
Helpers used by the rest of the notebook:
- `load_bug(bug_id)` reads a bug's `buggy.py`, the target function and its tests.
- `run_tests(bug_id, source)` copies the bug folder to `work/`, writes `source` as `buggy.py`, and runs the official `test_bug.py` plus **our own extra tests** (`EXTRA_TESTS`: edge cases such as empty input, cycles and unreachable nodes). Each run has a timeout so an infinite loop counts as a failure instead of hanging the notebook.
- `REQUIREMENTS` describes the expected behaviour of each function, taken from its docstring. `HINTS` points at the suspicious code and is used only in the escalated prompts (`tests-aware + hint`, `self-refine`).

In [ ]:
import os, shutil, subprocess, sys

WORK_DIR = Path("work")   # scratch copies for testing; data/ is never modified
TEST_TIMEOUT = 30         # seconds per pytest run; buggy loops can spin forever

REQUIREMENTS = {
    "BUG000": "Return True if any two different elements of `numbers` are closer to each other than "
              "`threshold` (their absolute difference is less than threshold), otherwise False.",
    "BUG001": "Dijkstra's algorithm: return the length of the shortest path from startnode to goalnode. "
              "The tentative distance of nextnode is the smaller of its current distance and "
              "(distance of the current node + length of edge (node, nextnode)). "
              "Return 0 when startnode is goalnode and float('inf') when goalnode is unreachable.",
    "BUG002": "Bellman-Ford: return a dict mapping every node to the minimum total weight of a path from "
              "source. Each round relaxes every edge (u, v) by lowering the weight of node v. "
              "Negative edges are allowed. Do not modify the input dict weight_by_edge.",
    "BUG003": "Kruskal's algorithm: return the set of edges of a minimum spanning tree. When an edge joins "
              "two components, every node of the merged component must refer to the same merged group.",
    "BUG004": "Floyd-Warshall: return a dict with the shortest path length for every ordered pair (i, j) "
              "of nodes 0..n-1. A path from i to j through k has length (i to k) + (k to j).",
    "BUG005": "Breadth-first search: return True if goalnode can be reached from startnode by following "
              "node.successors, otherwise False. The search starts at startnode and returns False "
              "once the queue is empty.",
    "BUG006": "Depth-first search: return True if goalnode can be reached from startnode by following "
              "node.successors, otherwise False. Each node is visited at most once so cycles terminate.",
    "BUG007": "Reverse a singly linked list (nodes linked by .successor) in place and return the new head. "
              "Return None for an empty list.",
    "BUG008": "Tortoise-and-hare cycle detection: return True if the linked list starting at node "
              "(linked by .successor) has a cycle, otherwise False. The tortoise moves 1 step and the "
              "hare moves 2 steps per iteration; return False as soon as the hare reaches the end.",
    "BUG009": "Return the length of the longest valid (well-formed) parentheses substring of s. "
              "Each matched pair '()' adds 2 to the length, e.g. '()' -> 2, '(())' -> 4, ')()())' -> 4.",
}

HINTS = {
    "BUG000": "The distance between two numbers must never be negative: compare the absolute difference with threshold.",
    "BUG001": "Look at the second argument of min(): the new distance to nextnode must start from `distance` "
              "(the distance of the node just popped), not from get(unvisited_nodes, nextnode).",
    "BUG002": "The relaxation result is stored in the wrong dictionary: it must update weight_by_node[v], "
              "not weight_by_edge[u, v].",
    "BUG003": "In the inner loop, `group_by_node[node].update(...)` is wrong: every node of the merged group "
              "must point to the same set object, i.e. group_by_node[node] = group_by_node[u].",
    "BUG004": "Check the indices of the second term inside min(...): the path from k to j is length_by_path[k, j].",
    "BUG005": "Two problems: the queue starts with the wrong node (it must start with startnode), "
              "and `while True` must instead loop only while the queue is not empty.",
    "BUG006": "Two problems: the node is never added to nodesvisited, and the generator iterates over "
              "`node` instead of `node.successors`.",
    "BUG007": "Inside the loop prevnode is never updated: set prevnode = node before moving to nextnode.",
    "BUG008": "The hare must move two steps (hare = hare.successor.successor), so check that both hare and "
              "hare.successor are not None before moving it; loop with `while True`.",
    "BUG009": "Each matched pair adds 2, not 1 (in both branches), and the elif condition must use `and` "
              "instead of `or`: the index must be valid AND the character there must be '('.",
}

# Our own robustness tests (the assignment asks for these). They run next to test_bug.py.
EXTRA_TESTS = {
    "BUG000": """from .buggy import has_close_elements

def test_extra_descending_values():
    # a signed difference would be negative here and wrongly count as "close"
    assert has_close_elements([5.0, 1.0, 9.0], 1.0) == False

def test_extra_close_pair():
    assert has_close_elements([1.0, 2.0, 1.5], 0.6) == True

def test_extra_empty_and_single():
    assert has_close_elements([], 1.0) == False
    assert has_close_elements([3.0], 1.0) == False
""",
    "BUG001": """from .node import Node
from .buggy import shortest_path_length

def test_extra_cheaper_two_hop_path():
    c = Node("c", None, [])
    b = Node("b", None, [c])
    a = Node("a", None, [b, c])
    assert shortest_path_length({(a, b): 1, (b, c): 1, (a, c): 5}, a, c) == 2

def test_extra_unreachable():
    a = Node("a", None, [])
    b = Node("b", None, [])
    assert shortest_path_length({}, a, b) == float("inf")
""",
    "BUG002": """from .buggy import shortest_paths

def test_extra_negative_edge_and_input_untouched():
    graph = {("A", "B"): 5, ("B", "C"): -3, ("A", "C"): 4}
    snapshot = dict(graph)
    assert shortest_paths("A", graph) == {"A": 0, "B": 5, "C": 2}
    assert graph == snapshot

def test_extra_two_hop_shortcut():
    graph = {("A", "B"): 1, ("B", "C"): 1, ("A", "C"): 5}
    assert shortest_paths("A", graph) == {"A": 0, "B": 1, "C": 2}
""",
    "BUG003": """from .buggy import minimum_spanning_tree

def test_extra_triangle():
    assert minimum_spanning_tree({(1, 2): 1, (2, 3): 2, (1, 3): 3}) == {(1, 2), (2, 3)}

def test_extra_merging_components():
    result = minimum_spanning_tree({(1, 2): 1, (3, 4): 1, (4, 5): 1, (2, 3): 2, (1, 4): 3, (1, 5): 5})
    assert result == {(1, 2), (3, 4), (4, 5), (2, 3)}
""",
    "BUG004": """from .buggy import shortest_path_lengths

def test_extra_shortcut_through_middle_node():
    result = shortest_path_lengths(3, {(0, 1): 1, (1, 2): 1, (0, 2): 5})
    assert result[0, 2] == 2
    assert result[0, 1] == 1 and result[1, 2] == 1
    assert result[2, 0] == float("inf")
    assert all(result[i, i] == 0 for i in range(3))
""",
    "BUG005": """from .node import Node
from .buggy import breadth_first_search

def test_extra_start_is_goal():
    a = Node("a", None, [])
    assert breadth_first_search(a, a)

def test_extra_direction_matters():
    c = Node("c", None, [])
    b = Node("b", None, [c])
    a = Node("a", None, [b])
    assert breadth_first_search(a, c)
    assert not breadth_first_search(c, a)

def test_extra_cycle_without_goal_terminates():
    a = Node("a", None, [])
    b = Node("b", None, [a])
    a.successors = [b]
    assert not breadth_first_search(a, Node("c", None, []))
""",
    "BUG006": """from .node import Node
from .buggy import depth_first_search

def test_extra_start_is_goal():
    a = Node("a", None, [])
    assert depth_first_search(a, a)

def test_extra_direction_matters():
    c = Node("c", None, [])
    b = Node("b", None, [c])
    a = Node("a", None, [b])
    assert depth_first_search(a, c)
    assert not depth_first_search(c, a)

def test_extra_cycle_without_goal_terminates():
    a = Node("a", None, [])
    b = Node("b", None, [a])
    a.successors = [b]
    assert not depth_first_search(a, Node("c", None, []))
""",
    "BUG007": """from .node import Node
from .buggy import reverse_linked_list

def test_extra_two_nodes():
    tail = Node("b")
    head = Node("a", tail)
    assert reverse_linked_list(head) is tail
    assert tail.successor is head
    assert head.successor is None
""",
    "BUG008": """from .node import Node
from .buggy import detect_cycle

def _chain(n):
    nodes = [Node(i) for i in range(n)]
    for a, b in zip(nodes, nodes[1:]):
        a.successor = b
    return nodes

def test_extra_single_node():
    assert not detect_cycle(Node(1))

def test_extra_self_loop():
    n = Node(1)
    n.successor = n
    assert detect_cycle(n)

def test_extra_even_length_without_cycle():
    assert not detect_cycle(_chain(4)[0])

def test_extra_cycle_in_tail():
    nodes = _chain(4)
    nodes[3].successor = nodes[1]
    assert detect_cycle(nodes[0])
""",
    "BUG009": """from .buggy import longestValidParentheses

def test_extra_empty():
    assert longestValidParentheses("") == 0

def test_extra_no_valid_pair():
    assert longestValidParentheses(")(") == 0

def test_extra_unclosed_prefix():
    assert longestValidParentheses("(()") == 2

def test_extra_leading_close():
    assert longestValidParentheses(")()())") == 4

def test_extra_adjacent_and_nested():
    assert longestValidParentheses("()(())") == 6
""",
}


def load_bug(bug_id: str) -> dict:
    source = (DATA_DIR / bug_id / "buggy.py").read_text(encoding="utf-8")
    func_name = TARGET_FUNCTION[bug_id]
    tree = ast.parse(source)
    fn = _function_node(tree, func_name)
    helpers = [ast.get_source_segment(source, n) for n in tree.body
               if isinstance(n, ast.FunctionDef) and n.name != func_name]
    return {
        "bug_id": bug_id,
        "func_name": func_name,
        "source": source,
        "func_src": "\n".join(source.splitlines()[fn.lineno - 1:fn.end_lineno]),
        "helpers": helpers,
        "tests": (DATA_DIR / bug_id / "test_bug.py").read_text(encoding="utf-8"),
        "requirement": REQUIREMENTS[bug_id],
        "hint": HINTS[bug_id],
    }


def _pytest(test_file: Path) -> dict:
    total = len(re.findall(r"^def test", test_file.read_text(), re.M))
    cmd = [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--tb=line", str(test_file)]
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=TEST_TIMEOUT,
                              env={**os.environ, "COLUMNS": "200"})  # don't truncate FAILED lines
    except subprocess.TimeoutExpired:
        return {"passed": 0, "total": total,
                "log": f"TIMEOUT: {test_file.name} did not finish in {TEST_TIMEOUT}s (infinite loop?)"}
    log = re.sub(r"\S*/(test_\w+\.py)", r"\1", proc.stdout + proc.stderr)
    m = re.search(r"(\d+) passed", log)
    return {"passed": int(m.group(1)) if m else 0, "total": total, "log": log}


def run_tests(bug_id: str, source: str) -> dict:
    """Test `source` as that bug's buggy.py against test_bug.py and our extra tests."""
    dest = WORK_DIR / bug_id
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(DATA_DIR / bug_id, dest,
                    ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache", ".DS_Store"))
    (dest / "buggy.py").write_text(source, encoding="utf-8")
    (dest / "test_extra.py").write_text(EXTRA_TESTS[bug_id], encoding="utf-8")
    return {"official": _pytest(dest / "test_bug.py"), "extra": _pytest(dest / "test_extra.py")}


def failure_summary(result: dict, max_lines: int = 12) -> str:
    """One line per failing test (with its assert message), error or timeout, to show the model."""
    lines = []
    for part in ("official", "extra"):
        for line in result[part]["log"].splitlines():
            if re.match(r"(FAILED|ERROR|TIMEOUT)", line) and line not in lines:
                lines.append(line.strip())
    return "\n".join(lines[:max_lines]) or "(no failure details)"


# Sanity check: every original buggy.py should fail at least one official test.
for bug_id in TARGET_FUNCTION:
    r = run_tests(bug_id, load_bug(bug_id)["source"])
    print(f"{bug_id} original: official {r['official']['passed']}/{r['official']['total']}, "
          f"extra {r['extra']['passed']}/{r['extra']['total']}")


### Prompt builders
One builder per technique, so every bug gets prompts with the same structure:

| Technique | What the prompt contains |
|---|---|
| `zero-shot` | buggy function + requirement + strict output format |
| `one-shot` | one worked buggy → fixed example, then the task |
| `chain-of-thought` | asks for 2-4 bullet points of reasoning, then the code |
| `few-shot` | three buggy → fixed examples (wrong operator, wrong variable, missing statement) |
| `tests-aware` | adds `test_bug.py` and the current pytest failures |
| `tests-aware + hint` | the tests-aware prompt plus a hint that points at the faulty code |
| `self-refine` | the model's own previous attempt + the tests it still fails, asking it to fix its code |

Helper functions that exist in `buggy.py` (e.g. BUG001's `get` / `insert_or_update`) are shown as read-only context.

In [ ]:
SYSTEM_LINE = "You are a helpful coding assistant."

def _output_rule(bug):
    return (f"Output: Return only the corrected `{bug['func_name']}` function (same name and parameters) "
            "in a single ```python code block. Do not add explanations or tests.")

def _helpers_block(bug):
    if not bug["helpers"]:
        return ""
    return ("These helper functions already exist and are correct (do not change them):\n"
            "```python\n" + "\n\n".join(bug["helpers"]) + "\n```\n")

def _task_block(bug):
    return (f"Function:\n```python\n{bug['func_src']}\n```\n"
            f"{_helpers_block(bug)}"
            f"Requirement: {bug['requirement']}\n")


def build_zero_shot(bug):
    return (f"{SYSTEM_LINE}\n\n"
            f"Fix the bug in this Python function.\n"
            f"{_task_block(bug)}"
            f"{_output_rule(bug)}\n")


ONE_SHOT_EXAMPLE = """Example (Buggy → Fixed)
Problem: find_max(nums) should return the largest element, but it returns the smallest.
Buggy:
```python
def find_max(nums):
    best = nums[0]
    for x in nums:
        if x < best:
            best = x
    return best
```
Fixed:
```python
def find_max(nums):
    best = nums[0]
    for x in nums:
        if x > best:
            best = x
    return best
```
"""

def build_one_shot(bug):
    return (f"{SYSTEM_LINE}\n\n"
            f"{ONE_SHOT_EXAMPLE}\n"
            f"Now fix a new function (One-Shot):\n"
            f"{_task_block(bug)}"
            f"{_output_rule(bug)}\n")


def build_cot(bug):
    return (f"{SYSTEM_LINE}\n\n"
            f"Task: Fix the bug in this function.\n"
            f"{_task_block(bug)}\n"
            f"Think step by step in 2-4 short bullet points: what the function should do, "
            f"which line is wrong, and how to fix it.\n"
            f"After your reasoning, output ONLY the final corrected `{bug['func_name']}` function "
            f"in a ```python code block and nothing else.\n")


FEW_SHOT_EXAMPLES = """Here are three examples (Buggy → Fixed):

Problem: wrong comparison operator. find_max(nums) should return the largest element.
Buggy:
```python
def find_max(nums):
    best = nums[0]
    for x in nums:
        if x < best:
            best = x
    return best
```
Fixed:
```python
def find_max(nums):
    best = nums[0]
    for x in nums:
        if x > best:
            best = x
    return best
```

Problem: wrong variable. total(prices) should return the sum of prices.
Buggy:
```python
def total(prices):
    result = 0
    for p in prices:
        result = p + p
    return result
```
Fixed:
```python
def total(prices):
    result = 0
    for p in prices:
        result = result + p
    return result
```

Problem: missing statement. count_down(n) should return [n, n-1, ..., 1].
Buggy:
```python
def count_down(n):
    out = []
    while n > 0:
        out.append(n)
    return out
```
Fixed:
```python
def count_down(n):
    out = []
    while n > 0:
        out.append(n)
        n -= 1
    return out
```
"""

def build_few_shot(bug):
    return (f"{SYSTEM_LINE}\n\n"
            f"{FEW_SHOT_EXAMPLES}\n"
            f"Now fix a new function:\n"
            f"{_task_block(bug)}"
            f"{_output_rule(bug)}\n")


_BASELINE = {}  # bug_id -> pytest result of the original buggy.py

def _baseline_failures(bug):
    if bug["bug_id"] not in _BASELINE:
        _BASELINE[bug["bug_id"]] = run_tests(bug["bug_id"], bug["source"])
    return failure_summary(_BASELINE[bug["bug_id"]])

def build_tests_aware(bug, hint=False):
    return (f"{SYSTEM_LINE}\n\n"
            f"The function below fails its unit tests. Fix it so that all tests pass.\n"
            f"{_task_block(bug)}"
            f"Unit tests:\n```python\n{bug['tests'].strip()}\n```\n"
            f"Current pytest failures:\n{_baseline_failures(bug)}\n"
            + (f"Hint: {bug['hint']}\n" if hint else "")
            + f"{_output_rule(bug)}\n")

def build_tests_aware_hint(bug):
    return build_tests_aware(bug, hint=True)


def build_self_refine(bug, previous_code, failures):
    return (f"{SYSTEM_LINE}\n\n"
            f"You tried to fix `{bug['func_name']}` and wrote this code:\n"
            f"```python\n{previous_code}\n```\n"
            f"{_helpers_block(bug)}"
            f"It still fails these tests:\n{failures}\n"
            f"Requirement: {bug['requirement']}\n"
            f"Hint: {bug['hint']}\n"
            f"Review your code line by line, find what is still wrong, and fix it.\n"
            f"{_output_rule(bug)}\n")


print(build_zero_shot(load_bug("BUG000")))


### Run, log, test and keep the best patch
`try_prompt` is the single entry point for every model call in this notebook:
1. call Gemma (`generate_text`)
2. log the attempt to `results/prompts.jsonl` (technique, prompt, completion, bug file, model)
3. extract the code and merge it into the full original `buggy.py`
4. run `test_bug.py` and the extra tests
5. if this candidate scores better than the current patch for that bug, save it with `save_final_patch`

A candidate's score is (official tests passed, extra tests passed). A bug counts as solved when both suites pass completely.

In [ ]:
MAX_NEW_TOKENS = 512
BEST = {}  # bug_id -> best attempt so far (the one saved in results/final_patches)

def _score(attempt):
    if not attempt or not attempt.get("result"):
        return (-1, -1)
    r = attempt["result"]
    return (r["official"]["passed"], r["extra"]["passed"])

def is_solved(attempt) -> bool:
    if not attempt or not attempt.get("result"):
        return False
    r = attempt["result"]
    return all(r[p]["passed"] == r[p]["total"] for p in ("official", "extra"))

def current_best(bug_id):
    """Best patch so far, including one saved by an earlier run of this notebook."""
    if bug_id not in BEST:
        saved = PATCH_DIR / f"{bug_id}.py"
        if saved.exists():
            source = saved.read_text(encoding="utf-8")
            BEST[bug_id] = {"technique": "(saved earlier)", "output": None, "merged": source,
                            "code": None, "result": run_tests(bug_id, source)}
    return BEST.get(bug_id)

def _fmt(result):
    if result is None:
        return "no usable code in the response"
    o, e = result["official"], result["extra"]
    return f"official {o['passed']}/{o['total']}, extra {e['passed']}/{e['total']}"

def try_prompt(bug_id: str, technique: str, prompt: str, do_sample: bool = False,
               max_new_tokens: int = MAX_NEW_TOKENS, show_output: bool = False) -> dict:
    bug = load_bug(bug_id)
    output = generate_text(prompt, max_new_tokens=max_new_tokens, do_sample=do_sample)
    log_prompt(technique, prompt, completion=output, bug_file=f"{bug_id}.py", model=MODEL_ID)

    code = extract_code(output, bug["func_name"])
    merged = merge_into_original(bug["source"], code, bug["func_name"]) if code else None
    result = run_tests(bug_id, merged) if merged else None
    attempt = {"technique": technique, "output": output, "code": code, "merged": merged, "result": result}

    saved = ""
    if result is not None and _score(attempt) > _score(current_best(bug_id)):
        BEST[bug_id] = attempt
        save_final_patch(f"{bug_id}.py", output)
        saved = "  -> saved as final patch"
    print(f"{bug_id} | {technique:<18} | {'sampled' if do_sample else 'greedy ':<7} | {_fmt(result)}{saved}")
    if show_output:
        print(output[:1500])
    return attempt


## 1. Zero‑Shot
A prompting technique that asks the model to solve the task **without** examples. Include specific output constraints.

In [ ]:
example_zero_shot_prompt = """You are a helpful coding assistant.

Fix the bug in this function.
Function:
```python
def add_one(x):
    return x - 1
```
Requirement: Make it return x + 1.
Output: Return only the corrected function in a Python code block, nothing else.
"""

print("----- ZERO-SHOT EXAMPLE PROMPT -----\n", example_zero_shot_prompt)
example_zero_shot_output = generate_text(example_zero_shot_prompt, max_new_tokens=120)
print("\n----- ZERO-SHOT EXAMPLE OUTPUT -----\n", example_zero_shot_output[:1000])


In [ ]:
# For example, your function calls should save the log and the final patch as shown below.
# They are commented out so the toy add_one example does not end up in the submission:
# the autograder expects only BUG000.py ... BUG009.py. Every real attempt in this notebook
# goes through try_prompt(), which calls log_prompt() and save_final_patch() itself.
# log_prompt("zero-shot", example_zero_shot_prompt, completion=example_zero_shot_output, bug_file="BUG_xxx.py", model="google/gemma-3-270m-it")
# save_final_patch("BUG_xxx.py", example_zero_shot_output)


In [ ]:
# Your turn: write a zero-shot prompt below  (BUG000 - has_close_elements)
your_zero_shot_prompt = build_zero_shot(load_bug("BUG000"))

print("\n----- YOUR ZERO-SHOT PROMPT -----\n", your_zero_shot_prompt)
attempt = try_prompt("BUG000", "zero-shot", your_zero_shot_prompt, max_new_tokens=200)
your_zero_shot_output = attempt["output"]
print("\n----- YOUR ZERO-SHOT OUTPUT -----\n", your_zero_shot_output[:1000])


## 2. One‑Shot
A prompting technique that includes **one worked example** before the target task.

In [ ]:
example_one_shot_prompt = """You are a helpful coding assistant.

Example (Problem → Solution)
Problem: Write a Python function add_one(x) that returns x + 1.
Solution:
```python
def add_one(x):
    return x + 1
```

Now fix a new function (One-Shot):
Function:
```python
def mul_two(a, b):
    return a + b
```
Requirement: Make it return a * b.
Output: Return only the corrected function in a Python code block.
"""

print("----- ONE-SHOT EXAMPLE PROMPT -----\n", example_one_shot_prompt)
example_one_shot_output = generate_text(example_one_shot_prompt, max_new_tokens=160)
print("\n----- ONE-SHOT EXAMPLE OUTPUT -----\n", example_one_shot_output[:1000])



In [ ]:
# Your turn: write a one-shot prompt below  (BUG007 - reverse_linked_list)
your_one_shot_prompt = build_one_shot(load_bug("BUG007"))

print("\n----- YOUR ONE-SHOT PROMPT -----\n", your_one_shot_prompt)
attempt = try_prompt("BUG007", "one-shot", your_one_shot_prompt, max_new_tokens=220)
your_one_shot_output = attempt["output"]
print("\n----- YOUR ONE-SHOT OUTPUT -----\n", your_one_shot_output[:1000])


## 3. Chain‑of‑Thought (CoT)
A prompting technique that asks the model for a brief high‑level plan (step-by-step) before the final code.

In [ ]:
cot_example_prompt = """You are a helpful coding assistant.

Task: Fix the bug in this function.
Function:
```python
def is_even(n):
    return n % 2 == 1
```

Think step by step in 2-4 short bullet points about the bug and the fix.
After your reasoning, output ONLY the final corrected function in a Python code block and nothing else.
"""

print("----- COT EXAMPLE PROMPT -----\n", cot_example_prompt)
cot_example_output = generate_text(cot_example_prompt, max_new_tokens=220)
print("\n----- COT EXAMPLE OUTPUT -----\n", cot_example_output[:1000])



In [ ]:
# Your turn: write a CoT prompt below  (BUG004 - Floyd-Warshall)
your_cot_prompt = build_cot(load_bug("BUG004"))

print("\n----- YOUR COT PROMPT -----\n", your_cot_prompt)
attempt = try_prompt("BUG004", "chain-of-thought", your_cot_prompt, max_new_tokens=400)
your_cot_output = attempt["output"]
print("\n----- YOUR COT OUTPUT -----\n", your_cot_output[:1000])


## 4. Few‑Shot
A prompting technique that provides multiple examples to shape the model's behavior.

In [ ]:
few_shot_example_prompt = """You are a helpful coding assistant.

Here are two examples (Problem → Solution):

Problem: Write a function mul_two(a, b) that returns a * b.
Solution:
```python
def mul_two(a, b):
    return a * b
```

Problem: Return the larger of two numbers.
Solution:
```python
def max_of_two(a, b):
    return a if a >= b else b
```

Now fix a new function:
Function:
```python
def square(x):
    return x + x
```
Requirement: Make it return x * x.
Output: Return only the corrected function in a Python code block.
"""

print("----- FEW-SHOT EXAMPLE PROMPT -----\n", few_shot_example_prompt)
few_shot_example_output = generate_text(few_shot_example_prompt, max_new_tokens=200)
print("\n----- FEW-SHOT EXAMPLE OUTPUT -----\n", few_shot_example_output[:1000])



In [ ]:
# Your turn: write a few-shot prompt below  (BUG002 - Bellman-Ford)
your_few_shot_prompt = build_few_shot(load_bug("BUG002"))

print("\n----- YOUR FEW-SHOT PROMPT -----\n", your_few_shot_prompt)
attempt = try_prompt("BUG002", "few-shot", your_few_shot_prompt, max_new_tokens=260)
your_few_shot_output = attempt["output"]
print("\n----- YOUR FEW-SHOT OUTPUT -----\n", your_few_shot_output[:1000])


## 5. Tests-Aware, Self-Refine and the full repair loop
For every bug, `repair_bug` escalates through the techniques until a candidate passes **both** the official tests and our extra tests:

1. `zero-shot` → `one-shot` → `chain-of-thought` → `few-shot` → `tests-aware` → `tests-aware + hint`. Each prompt runs once greedily, then `SAMPLES_PER_TECHNIQUE - 1` more times with sampling, so the same prompt can yield different candidates.
2. `self-refine`: the best candidate so far and the tests it still fails go back to the model, for up to `SELF_REFINE_ROUNDS` rounds.

Every attempt is logged, successful or not. A bug already solved by an earlier cell or run is skipped unless `FORCE = True`. With a T4 the whole loop takes roughly 10-30 minutes, depending on how many bugs need escalation.

In [ ]:
SAMPLES_PER_TECHNIQUE = 3   # 1 greedy + 2 sampled attempts per technique
SELF_REFINE_ROUNDS = 4
FORCE = False               # True: keep prompting even for bugs that are already solved

PLAN = [
    ("zero-shot", build_zero_shot),
    ("one-shot", build_one_shot),
    ("chain-of-thought", build_cot),
    ("few-shot", build_few_shot),
    ("tests-aware", build_tests_aware),
    ("tests-aware + hint", build_tests_aware_hint),
]

def repair_bug(bug_id: str):
    if not FORCE and is_solved(current_best(bug_id)):
        print(f"{bug_id} | already solved ({current_best(bug_id)['technique']}), skipping")
        return current_best(bug_id)

    bug = load_bug(bug_id)
    best_this_run = None
    for technique, build in PLAN:
        for k in range(SAMPLES_PER_TECHNIQUE):
            attempt = try_prompt(bug_id, technique, build(bug), do_sample=k > 0)
            if attempt["result"] and _score(attempt) > _score(best_this_run):
                best_this_run = attempt
            if is_solved(attempt):
                return attempt

    for r in range(SELF_REFINE_ROUNDS):
        if best_this_run is None:
            break  # nothing parseable to refine
        prompt = build_self_refine(bug, best_this_run["code"], failure_summary(best_this_run["result"]))
        attempt = try_prompt(bug_id, "self-refine", prompt, do_sample=r > 0)
        if attempt["result"] and _score(attempt) >= _score(best_this_run):
            best_this_run = attempt
        if is_solved(attempt):
            return attempt
    return current_best(bug_id)


for bug_id in TARGET_FUNCTION:
    repair_bug(bug_id)
    print("-" * 80)


## 6. Line-level (fault-localized) repair
In the run above, Gemma 270M mostly returned the buggy function unchanged, even with hints, because a model this small struggles to rewrite a whole function. This step makes the task smaller: each prompt is a few-shot prompt showing **only the suspicious line** and a description of its problem, and asks for just that line rewritten. Bugs with several faulty lines get one prompt per line, and the candidate combinations are tested. Every call is logged as `line-level few-shot`, and a patch replaces the saved one only if it scores better.

In [ ]:
# 6. Line-level (fault-localized) repair
# Gemma 270M mostly copies a whole buggy function back unchanged. Here each prompt shows only the
# suspicious line plus a description of the problem, and asks for just that line rewritten.
# The fixed lines are put back into the original buggy.py and tested like every other attempt.
import itertools

LINE_SAMPLES = 5   # per line: 1 greedy + 4 sampled answers
MAX_COMBOS = 64    # bugs with several lines: test combinations of the candidates

LINE_FEW_SHOT = """You are a helpful coding assistant. You fix one line of Python at a time.

Example 1
Buggy line:
```python
if x < best:
```
Problem: the function should keep the largest value, so the comparison is reversed.
Fixed line:
```python
if x > best:
```

Example 2
Buggy line:
```python
total = p + p
```
Problem: the running total must add p to total, not p to itself.
Fixed line:
```python
total = total + p
```

Example 3
Buggy line:
```python
out.append(n)
```
Problem: n is never decreased. Rewrite the line followed by the missing line that lowers n by 1.
Fixed line:
```python
out.append(n)
n -= 1
```
"""

# (exact buggy line, description of the problem) for each bug
LINE_TASKS = {
    "BUG000": [("distance = elem - elem2",
                "the distance between two numbers must never be negative; use the absolute value of the difference.")],
    "BUG001": [("get(unvisited_nodes, nextnode) + length_by_edge[node, nextnode]",
                "the new distance to nextnode must be `distance` (the distance of the current node) plus the "
                "edge length, not get(unvisited_nodes, nextnode) plus the edge length.")],
    "BUG002": [("weight_by_edge[u, v] = min(",
                "the relaxed weight must be stored in weight_by_node[v]; the input dict weight_by_edge must not change.")],
    "BUG003": [("group_by_node[node].update(group_by_node[u])",
                "every node of the merged group must point to the same set object as u: assign "
                "group_by_node[u] to group_by_node[node] instead of calling update.")],
    "BUG004": [("length_by_path[i, k] + length_by_path[j, k]",
                "a path from i to j through k is (i to k) + (k to j), so the second term must be length_by_path[k, j].")],
    "BUG005": [("queue.append(goalnode)",
                "breadth-first search must start from startnode, so startnode is the first node put in the queue."),
               ("while True:",
                "the loop must stop when the queue is empty, so it should loop only while queue is not empty.")],
    "BUG006": [("else:",
                "in the else branch the current node must first be added to the set nodesvisited. "
                "Rewrite the line followed by the missing line."),
               ("search_from(nextnode) for nextnode in node",
                "the search must iterate over node.successors, not over node itself.")],
    "BUG007": [("node.successor = prevnode",
                "after reversing the pointer, prevnode must be set to the current node. "
                "Rewrite the line followed by the missing line.")],
    "BUG008": [("if hare is None:",
                "the hare moves two steps, so return False when hare is None or hare.successor is None."),
               ("hare = hare.successor",
                "the hare must move two steps per iteration, i.e. hare.successor.successor.")],
    "BUG009": [("dp[i] = (dp[i - 2] if i >= 2 else 0) + 1",
                "a matched pair '()' has length 2, so add 2 instead of 1."),
               ('elif i - dp[i - 1] > 0 or s[i - dp[i - 1] - 1] == "(":',
                "both conditions must hold (the index is valid AND the character there is '('), "
                "so use `and` instead of `or`."),
               ("+ 1",
                "this term counts the matched pair, which has length 2, so it must be + 2.")],
}


def build_line_prompt(buggy_line, problem):
    return (f"{LINE_FEW_SHOT}\n"
            f"Now fix this line.\n"
            f"Buggy line:\n```python\n{buggy_line}\n```\n"
            f"Problem: {problem}\n"
            f"Output only the fixed line in a ```python code block.\n")


def _parse_line_fix(output, buggy_line):
    """The rewritten line(s) from a model answer, or None if it gave nothing new."""
    blocks = _CODE_BLOCK.findall(output)
    text = textwrap.dedent(blocks[0] if blocks else output)
    lines = [l.rstrip() for l in text.splitlines()
             if l.strip() and not re.match(r"\s*(```|#|Fixed|Buggy|Problem|Here|Output)", l)]
    lines = lines[:2]
    if not lines or (len(lines) == 1 and lines[0].strip() == buggy_line.strip()):
        return None
    return tuple(lines)


def _apply_line_fixes(source, fixes):
    """Replace each buggy line (matched exactly once, ignoring indentation) with its fix."""
    lines = source.splitlines()
    for buggy, new in fixes:
        hits = [i for i, l in enumerate(lines) if l.strip() == buggy.strip()]
        if len(hits) != 1:
            return None
        i = hits[0]
        indent = lines[i][:len(lines[i]) - len(lines[i].lstrip())]
        base = len(new[0]) - len(new[0].lstrip())
        out = [indent + new[0].lstrip()]
        for extra in new[1:]:
            rel = max(len(extra) - len(extra.lstrip()) - base, 0)
            if new[0].rstrip().endswith(":") and rel == 0:
                rel = 4  # body of an `else:` / `if ...:` the model left unindented
            out.append(indent + " " * rel + extra.lstrip())
        lines[i:i + 1] = out
    patched = "\n".join(lines) + "\n"
    try:
        ast.parse(patched)
    except SyntaxError:
        return None
    return patched


def line_level_repair(bug_id):
    if not FORCE and is_solved(current_best(bug_id)):
        print(f"{bug_id} | already solved ({current_best(bug_id)['technique']}), skipping")
        return
    bug = load_bug(bug_id)
    options = []
    for buggy, problem in LINE_TASKS[bug_id]:
        prompt = build_line_prompt(buggy, problem)
        candidates = []
        for k in range(LINE_SAMPLES):
            out = generate_text(prompt, max_new_tokens=80, do_sample=k > 0)
            log_prompt("line-level few-shot", prompt, completion=out, bug_file=f"{bug_id}.py", model=MODEL_ID)
            fix = _parse_line_fix(out, buggy)
            if fix and fix not in candidates:
                candidates.append(fix)
        print(f"{bug_id} | line `{buggy}` -> {[' / '.join(c) for c in candidates] or 'no new candidates'}")
        options.append([(buggy, c) for c in candidates] or [(buggy, (buggy,))])

    for combo in itertools.islice(itertools.product(*options), MAX_COMBOS):
        source = _apply_line_fixes(bug["source"], combo)
        if source is None:
            continue
        attempt = {"technique": "line-level few-shot", "output": None, "code": None,
                   "merged": source, "result": run_tests(bug_id, source)}
        if _score(attempt) > _score(current_best(bug_id)):
            BEST[bug_id] = attempt
            save_final_patch(f"{bug_id}.py", f"```python\n{source}```")
        if is_solved(attempt):
            break
    print(f"{bug_id} | best: {_fmt(current_best(bug_id)['result'])}"
          f"{'  SOLVED' if is_solved(current_best(bug_id)) else ''}")


for bug_id in TARGET_FUNCTION:
    line_level_repair(bug_id)
    print("-" * 80)


### Optional: try your own prompt for a bug that is still failing
If the summary below shows a bug that is not solved, write a prompt by hand here (or tweak `REQUIREMENTS` / `HINTS` above) and run it with `try_prompt`. It is logged like every other attempt, and it replaces the saved patch only if it scores better.

In [ ]:
# Example: a hand-written tests-aware prompt. Change BUG_ID / the text and re-run as often as you like.
BUG_ID = "BUG009"
bug = load_bug(BUG_ID)
my_prompt = f"""You are a helpful coding assistant.

Fix this function. It must return 2 for "()", 4 for "(())" and 4 for "()()".
```python
{bug['func_src']}
```
Hint: {bug['hint']}
Output: Return only the corrected `{bug['func_name']}` function in a ```python code block.
"""
# Uncomment to run:
# try_prompt(BUG_ID, "tests-aware", my_prompt, show_output=True)


## Results summary
Re-tests every saved patch and shows which technique produced it.

In [ ]:
rows = []
for bug_id in TARGET_FUNCTION:
    best = current_best(bug_id)
    rows.append((bug_id, best["technique"] if best else "-", _fmt(best["result"]) if best else "NO PATCH",
                 "yes" if is_solved(best) else "NO"))

n_attempts = sum(1 for _ in PROMPTS_PATH.open(encoding="utf-8")) if PROMPTS_PATH.exists() else 0
print(f"{'bug':<8}{'technique':<20}{'tests':<34}solved")
for r in rows:
    print(f"{r[0]:<8}{r[1]:<20}{r[2]:<34}{r[3]}")
print(f"\n{n_attempts} attempts logged in {PROMPTS_PATH}")


## Build the submission ZIP
Checks the required structure and writes `SWE270P_A1.ZIP` containing `results/prompts.jsonl` and `results/final_patches/BUG000.py … BUG009.py`.

In [ ]:
import zipfile

expected = [f"BUG{i:03d}.py" for i in range(10)]

# Remove anything else (e.g. BUG_xxx.py from the template example) so the autograder sees only the 10 patches.
for p in PATCH_DIR.iterdir():
    if p.name not in expected:
        print("Removing", p)
        shutil.rmtree(p) if p.is_dir() else p.unlink()

missing = [n for n in expected if not (PATCH_DIR / n).exists()]
if missing:
    raise FileNotFoundError(f"Missing final patches: {missing}. Run the repair loop (or try_prompt) for those bugs.")
if not PROMPTS_PATH.exists() or PROMPTS_PATH.stat().st_size == 0:
    raise FileNotFoundError(f"{PROMPTS_PATH} is empty")

unsolved = [bug_id for bug_id in TARGET_FUNCTION if not is_solved(current_best(bug_id))]
if unsolved:
    print("WARNING: these patches do not pass all tests yet:", unsolved)

ZIP_NAME = "SWE270P_A1.ZIP"
with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("results/", "")
    zf.writestr("results/final_patches/", "")
    zf.write(PROMPTS_PATH, "results/prompts.jsonl")
    for name in expected:
        zf.write(PATCH_DIR / name, f"results/final_patches/{name}")

with zipfile.ZipFile(ZIP_NAME) as zf:
    print("\n".join(zf.namelist()))
print("\nWrote", Path(ZIP_NAME).resolve())

try:
    from google.colab import files
    files.download(ZIP_NAME)
except ImportError:
    pass  # running locally: the ZIP is in the working directory


## Submission checklist

Before you submit:
- Ensure `results/prompts.jsonl` contains all prompts you tried (one JSON line per attempt). <br> FYI: Make sure to check the rubric for full marks on this!
- Ensure you have one final repaired code for each buggy Python function, saved under `results/final_patches/`.
- Make sure each file is named exactly `BUGxxx.py` (e.g., `BUG001.py`).

What to upload:
- `results/prompts.jsonl`
- `results/final_patches/` folder containing all 10 repaired code files, one per buggy Python function

**Required File Structure**:  
Your submission ZIP file **must** follow this exact file structure for compatibility with the autograder:    
```
SWE270P_A1.ZIP
└── results
    ├── prompts.jsonl
    └── final_patches
        ├── BUG000.py
        ├── ...
        └── BUG009.py
```